# TC_12: CNN image perturbation
Apply blur and brightness perturbation. Run full diagnostic pipline and comare against NB10 CNN baseline.
UQ: Deep Ensemble, MC Dropout
XAI: Grad-CAM

In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from utils.visualization import plot_gradcam_sample
from torchvision import datasets, transforms


sys.path.append('..')
os.chdir('..')
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

from src.utils.config_loader import load_config
from src.utils.data_loader import load_image_data
from src.data_diagnostics.quality_checks import check_image_quality
from src.utils.visualization import plot_class_distribution, plot_correlation, plot_feature_boxplots
from src.utils.performance_tracker import PerformanceTracker
from src.models.image_models import train_simple_cnn, evaluate_image_model
from src.inference_diagnostics.uncertainty import mc_dropout_prediction_img, deep_ensemble_prediction_img
from src.inference_diagnostics.explainability import gradcam_img, collect_gradcam_feedback
from src.inference_diagnostics.flagging import assign_flags, evaluate_flags
from src.utils.visualization import plot_uncertainty_distribution, plot_flag_distribution
from src.utils.report_generator import generate_report,save_report
from src.data_diagnostics.perturbations import apply_blur_img, apply_brightness_img


config = load_config()
tracker = PerformanceTracker()

In [2]:
class BlurTransform:
    def __init__(self, sigma):
        self.sigma = sigma
    def __call__(self, img):
        return apply_blur_img(img, self.sigma)

class BrightnessTransform:
        def __init__(self, delta):
            self.delta = delta
        def __call__(self, img):
            return apply_brightness_img(img, self.delta)

def load_corrupted_image_data(config, perturbation_type, level):
    dataset_config = config['datasets']['image_data']
    image_size = dataset_config['image_size']
    data_path =  os.path.join(config['paths']['data_raw'], dataset_config['folder_name'])

    transform_list = [transforms.Resize((image_size, image_size))]

    if perturbation_type == 'blur':
        transform_list.append(BlurTransform(sigma = level))
    else:
        transform_list.append(BrightnessTransform(delta = level))

    transform_list.append(transforms.ToTensor())
    corrupted_transform = transforms.Compose(transform_list)

    train_set =  datasets.ImageFolder(
        root = os.path.join(data_path, 'train'),
        transform = corrupted_transform
    )

    return train_set




In [3]:
train_set, test_set = load_image_data(config)
seed = config['random_seeds']['primary_seed']
blur_levels = config['stage1_data_quality']['perturbations']['image_corruption']['blur_sigma']
brightness_level = config['stage1_data_quality']['perturbations']['image_corruption']['brightness_delta']

# MC and DE threshold values from NB10
mc_threshold = 0.0804
de_threshold = 0.1051

Loading dataset.
Loaded: 50000 train and 10000 test
Image size: 32
Classes: 10


### Blur perturbation

In [4]:
blur_result = {}

for sigma in blur_levels:

    # Load corrupted training data
    train_set_corrupted = load_corrupted_image_data(config, 'blur', sigma)

    tracker.start_performance_track()
    cnn_model = train_simple_cnn(train_set_corrupted, config)
    tracker.stop_performance_track(f"CNN training blur sigma {sigma}")

    tracker.start_performance_track()
    cnn_accuracy, cnn_prediction, cnn_report = evaluate_image_model(cnn_model, test_set, config)
    tracker.stop_performance_track(f"CNN evaluation blur sigma {sigma}")

    # MC Dropout
    tracker.start_performance_track()
    mc_means_probabilities, mc_uncertainty = mc_dropout_prediction_img(cnn_model, test_set, config)
    tracker.stop_performance_track(f"CNN MC Dropout blur sigma {sigma}")

    plot_uncertainty_distribution(mc_uncertainty, f"CNN MC Dropout blur sigma {sigma}", mc_threshold, config)

    # Deep Ensemble
    tracker.start_performance_track()
    de_means_probabilities, de_uncertainty = deep_ensemble_prediction_img(train_simple_cnn, config, test_set, train_set_corrupted  )
    tracker.stop_performance_track(f"CNN Deep Ensemble prediction sigma {sigma}")

    plot_uncertainty_distribution(de_uncertainty, f"CNN Deep Ensemble prediction sigma {sigma}", de_threshold, config)

    # Grad-CAM
    tracker.start_performance_track()
    heatmaps = gradcam_img(cnn_model, test_set, config)
    tracker.stop_performance_track(f"CNN Grad-CAM sigma {sigma}")

    # Show grid for expert review
    plot_gradcam_sample(test_set, heatmaps, 20, config, f"CNN blur sigma {sigma}", save = True)
    consistency_scores, correct, incorrect, partial = collect_gradcam_feedback(20)
    print(f"Human feedback: correct: {correct}, incorrect: {incorrect}, partial: {partial}")

    # Get true  labels for first 20 samples
    y_true_20 = np.array([test_set[i][1] for i in range(20)])

    # MC Dropout with 20 sample
    mc_y_pred_20 = mc_means_probabilities[:20].argmax(axis = 1)
    mc_flags_reviewed = assign_flags(mc_uncertainty[:20], consistency_scores, mc_threshold, 0.5)
    mc_flag_result_reviewed = evaluate_flags(mc_flags_reviewed, mc_y_pred_20, y_true_20)
    plot_flag_distribution(mc_flags_reviewed, "CNN MC + Grad-CAM Reviewed", config)

    # Deep Ensemble with 20 sample
    de_y_pred_20 = de_means_probabilities[:20].argmax(axis = 1)
    de_flags_reviewed = assign_flags(de_uncertainty[:20], consistency_scores, de_threshold, 0.5)
    de_flag_result_reviewed = evaluate_flags(de_flags_reviewed, de_y_pred_20, y_true_20)
    plot_flag_distribution(de_flags_reviewed, "CNN DE + Grad-CAM Reviewed", config)

    # 20 sample test set flagging (UQ only, no human review)
    fake_consistency_20 = np.ones(20)

    # MC only
    mc_flags_20 = assign_flags(mc_uncertainty[:20], fake_consistency_20, mc_threshold, 0.5)
    mc_20_result = evaluate_flags(mc_flags_20, mc_y_pred_20, y_true_20)
    plot_flag_distribution(mc_flags_20, "CNN MC UQ Only for 20", config)

    # Deep Ensemble only
    de_flags_20 = assign_flags(de_uncertainty[:20], fake_consistency_20, de_threshold, 0.5)
    de_20_result = evaluate_flags(de_flags_20, de_y_pred_20, y_true_20)
    plot_flag_distribution(de_flags_20, "CNN DE UQ Only for 20", config)

    # Generate report
    level_result = {
    'perturbation': 'blur',
    'model': 'Simple CNN',
    'accuracy': round(cnn_accuracy, 4),
    'classification_report': cnn_report,
    'mc_uncertainty': {
        'mean': round(float(mc_uncertainty.mean()), 8),
        'max': round(float(mc_uncertainty.max()), 8),
        'threshold': mc_threshold
    },
    'de_uncertainty': {
        'mean': round(float(de_uncertainty.mean()), 8),
        'max': round(float(de_uncertainty.max()), 8),
        'threshold': de_threshold
    },
    'gradcam_feedback':{
        'correct': int(correct),
        'incorrect': int(incorrect),
        'partial': int(partial)
    },
    'flagging_mc': mc_flag_result_reviewed,
    'flagging_de': de_flag_result_reviewed,
    'flagging_mc_20_mc_only': mc_20_result,
    'flagging_mc_20_de_only': de_20_result,
    'mc_vs_mc_plus_xai': round(mc_flag_result_reviewed['GREEN']['accuracy'] - mc_20_result['GREEN']['accuracy'], 4),
    'de_vs_de_plus_xai': round(de_flag_result_reviewed['GREEN']['accuracy'] - de_20_result['GREEN']['accuracy'], 4),
    'performance': tracker.get_results()
    }

    blur_result[f"sigma_{sigma}"] = level_result

    report_output = generate_report(config,'CIFAR-10 - CNN with blur ', stage2_result = level_result )
    save_report(report_output, f'TC_12_Simple_CNN_With_Blur_Sigma{sigma}.json', config)

Simple CNN training started.
Epoch 5/20
Epoch 10/20
Epoch 15/20
Epoch 20/20
Simple CNN training completed.
CNN training blur sigma 1.0: Time:433.92s, Memory:699.30MB
 Image model evaluation started.
{'0': {'precision': 0.4123581336696091, 'recall': 0.654, 'f1-score': 0.505800464037123, 'support': 1000.0}, '1': {'precision': 0.2857142857142857, 'recall': 0.906, 'f1-score': 0.43442819467753535, 'support': 1000.0}, '2': {'precision': 0.4444444444444444, 'recall': 0.204, 'f1-score': 0.2796435915010281, 'support': 1000.0}, '3': {'precision': 0.3204715969989282, 'recall': 0.299, 'f1-score': 0.3093636833936886, 'support': 1000.0}, '4': {'precision': 0.6333333333333333, 'recall': 0.076, 'f1-score': 0.1357142857142857, 'support': 1000.0}, '5': {'precision': 0.483065953654189, 'recall': 0.271, 'f1-score': 0.3472133247918001, 'support': 1000.0}, '6': {'precision': 0.403242147922999, 'recall': 0.398, 'f1-score': 0.40060392551585305, 'support': 1000.0}, '7': {'precision': 0.577023498694517, 'recall

In [5]:
brightness_result = {}

for delta in brightness_level:

    # Load corrupted training data
    train_set_corrupted = load_corrupted_image_data(config, 'brightness', delta)

    tracker.start_performance_track()
    cnn_model = train_simple_cnn(train_set_corrupted, config)
    tracker.stop_performance_track(f"CNN training brightness delta {delta}")

    tracker.start_performance_track()
    cnn_accuracy, cnn_prediction, cnn_report = evaluate_image_model(cnn_model, test_set, config)
    tracker.stop_performance_track(f"CNN evaluation brightness delta {delta}")

    # MC Dropout
    tracker.start_performance_track()
    mc_means_probabilities, mc_uncertainty = mc_dropout_prediction_img(cnn_model, test_set, config)
    tracker.stop_performance_track(f"CNN MC Dropout brightness delta {delta}")

    plot_uncertainty_distribution(mc_uncertainty, f"CNN MC Dropout brightness delta {delta}", mc_threshold, config)

    # Deep Ensemble
    tracker.start_performance_track()
    de_means_probabilities, de_uncertainty = deep_ensemble_prediction_img(train_simple_cnn, config, test_set, train_set_corrupted  )
    tracker.stop_performance_track(f"CNN Deep Ensemble prediction delta {delta}")

    plot_uncertainty_distribution(de_uncertainty, f"CNN Deep Ensemble prediction delta {delta}", de_threshold, config)

    # Grad-CAM
    tracker.start_performance_track()
    heatmaps = gradcam_img(cnn_model, test_set, config)
    tracker.stop_performance_track(f"CNN Grad-CAM delta {delta}")

    # Show grid for expert review
    plot_gradcam_sample(test_set, heatmaps, 20, config, f"CNN brightness delta {delta}", save = True)
    consistency_scores, correct, incorrect, partial = collect_gradcam_feedback(20)
    print(f"Human feedback: correct: {correct}, incorrect: {incorrect}, partial: {partial}")

    # Get true  labels for first 20 samples
    y_true_20 = np.array([test_set[i][1] for i in range(20)])

    # MC Dropout with 20 sample
    mc_y_pred_20 = mc_means_probabilities[:20].argmax(axis = 1)
    mc_flags_reviewed = assign_flags(mc_uncertainty[:20], consistency_scores, mc_threshold, 0.5)
    mc_flag_result_reviewed = evaluate_flags(mc_flags_reviewed, mc_y_pred_20, y_true_20)
    plot_flag_distribution(mc_flags_reviewed, "CNN MC + Grad-CAM Reviewed", config)

    # Deep Ensemble with 20 sample
    de_y_pred_20 = de_means_probabilities[:20].argmax(axis = 1)
    de_flags_reviewed = assign_flags(de_uncertainty[:20], consistency_scores, de_threshold, 0.5)
    de_flag_result_reviewed = evaluate_flags(de_flags_reviewed, de_y_pred_20, y_true_20)
    plot_flag_distribution(de_flags_reviewed, "CNN DE + Grad-CAM Reviewed", config)

    # 20 sample test set flagging (UQ only, no human review)
    fake_consistency_20 = np.ones(20)

    # MC only
    mc_flags_20 = assign_flags(mc_uncertainty[:20], fake_consistency_20, mc_threshold, 0.5)
    mc_20_result = evaluate_flags(mc_flags_20, mc_y_pred_20, y_true_20)
    plot_flag_distribution(mc_flags_20, "CNN MC UQ Only for 20", config)

    # Deep Ensemble only
    de_flags_20 = assign_flags(de_uncertainty[:20], fake_consistency_20, de_threshold, 0.5)
    de_20_result = evaluate_flags(de_flags_20, de_y_pred_20, y_true_20)
    plot_flag_distribution(de_flags_20, "CNN DE UQ Only for 20", config)

    # Generate report
    level_result = {
    'perturbation': 'brightness',
    'model': 'Simple CNN',
    'accuracy': round(cnn_accuracy, 4),
    'classification_report': cnn_report,
    'mc_uncertainty': {
        'mean': round(float(mc_uncertainty.mean()), 8),
        'max': round(float(mc_uncertainty.max()), 8),
        'threshold': mc_threshold
    },
    'de_uncertainty': {
        'mean': round(float(de_uncertainty.mean()), 8),
        'max': round(float(de_uncertainty.max()), 8),
        'threshold': de_threshold
    },
    'gradcam_feedback':{
        'correct': int(correct),
        'incorrect': int(incorrect),
        'partial': int(partial)
    },
    'flagging_mc': mc_flag_result_reviewed,
    'flagging_de': de_flag_result_reviewed,
    'flagging_mc_20_mc_only': mc_20_result,
    'flagging_mc_20_de_only': de_20_result,
    'mc_vs_mc_plus_xai': round(mc_flag_result_reviewed['GREEN']['accuracy'] - mc_20_result['GREEN']['accuracy'], 4),
    'de_vs_de_plus_xai': round(de_flag_result_reviewed['GREEN']['accuracy'] - de_20_result['GREEN']['accuracy'], 4),
    'performance': tracker.get_results()
    }

    brightness_result[f"delta_{delta}"] = level_result

    report_output = generate_report(config,'CIFAR-10 - CNN with brightness ', stage2_result = level_result )
    save_report(report_output, f'TC_12_Simple_CNN_With_Brightness_Delta{delta}.json', config)

Simple CNN training started.
Epoch 5/20
Epoch 10/20
Epoch 15/20
Epoch 20/20
Simple CNN training completed.
CNN training brightness delta 0.2: Time:360.85s, Memory:0.00MB
 Image model evaluation started.
{'0': {'precision': 0.7151162790697675, 'recall': 0.615, 'f1-score': 0.6612903225806451, 'support': 1000.0}, '1': {'precision': 0.5520195838433293, 'recall': 0.902, 'f1-score': 0.6848899012908124, 'support': 1000.0}, '2': {'precision': 0.6993987975951904, 'recall': 0.349, 'f1-score': 0.4656437625083389, 'support': 1000.0}, '3': {'precision': 0.4729119638826185, 'recall': 0.419, 'f1-score': 0.4443266171792153, 'support': 1000.0}, '4': {'precision': 0.694331983805668, 'recall': 0.343, 'f1-score': 0.45917001338688085, 'support': 1000.0}, '5': {'precision': 0.6014319809069213, 'recall': 0.504, 'f1-score': 0.5484221980413493, 'support': 1000.0}, '6': {'precision': 0.4645127118644068, 'recall': 0.877, 'f1-score': 0.6073407202216067, 'support': 1000.0}, '7': {'precision': 0.8372093023255814, '